In [1]:
# Imports
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
import numpy as np
import json
import re
import time
import sys
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report

sys.path.insert(0, 'training')
print("✓ Imports loaded")

✓ Imports loaded


In [2]:
# Load configuration and model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("./models/training_metadata_Thunderbird_Chunk1.json", "r") as f:
    metadata = json.load(f)
with open("./models/event_vocab_Thunderbird_Chunk1.json", "r") as f:
    event_vocab = json.load(f)

sequence_length = metadata['sequence_length']
block_size = metadata['block_size']

print(f"Device: {device}")
print(f"Sequence length: {sequence_length}, Block size: {block_size}, Vocab: {len(event_vocab)}")

Device: cuda
Sequence length: 80, Block size: 500, Vocab: 609


In [ ]:
# Model Architecture
class EventSequenceModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoder = nn.Embedding(vocab_size, embed_dim)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4, dropout=.3), num_layers=2)
        self.attn = nn.Linear(embed_dim, 1)
        self.fc = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim), nn.ReLU(), nn.Dropout(.3), nn.Linear(hidden_dim, 2))

    def forward(self, x):
        batch_size, seq_len = x.size()
        x = self.embed(x)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        x = x + self.pos_encoder(positions)
        x = self.transformer(x.permute(1, 0, 2))
        attn_weights = torch.softmax(self.attn(x), dim=0)
        return self.fc((x * attn_weights).sum(dim=0))

model = EventSequenceModel(metadata['vocab_size'], metadata['embed_dim'], metadata['hidden_dim'])
model.load_state_dict(torch.load("./models/TableVIII_thunderbird_realtime_classifier.pth", weights_only=True))
model.to(device).eval()
print("✓ Model loaded")

/home/sslab/miniconda3/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


✓ Model loaded


In [4]:
# QwenLog Deterministic Real-Time Parser for Thunderbird
import qwenlog_thunderbird_parser_bank as parser_bank

class QwenLogRealTimeParser:
    """Deterministic parser - sequential template matching (1→6663), first match wins"""
    
    def __init__(self, event_vocab):
        self.event_vocab = event_vocab
        self.unknown_idx = len(event_vocab) - 1
        self.all_parsers = []
        # Thunderbird log regex
        self.thunderbird_regex = re.compile(
            r'^(\S+)\s+'           # Label (- or alert type)
            r'(\d+)\s+'            # Timestamp
            r'(\d+\.\d+\.\d+)\s+'  # Date
            r'(\S+)\s+'            # Node
            r'(\w+\s+\d+)\s+'      # Month Day
            r'(\d+:\d+:\d+)\s+'    # Time
            r'(\S+)\s+'            # User
            r'(.*)$'               # Content
        )
        self._load_parsers()
        print(f"✓ Loaded {len(self.all_parsers)} templates")
    
    def _load_parsers(self):
        for i in range(1, 10000):  # Thunderbird has 6663 templates
            if hasattr(parser_bank, f'is_log_template_{i}'):
                self.all_parsers.append((i, getattr(parser_bank, f'is_log_template_{i}')))
    
    def parse_line(self, line):
        match = self.thunderbird_regex.match(line.strip())
        if not match:
            return None, None
        label_str = match.group(1)
        content = match.group(8)
        
        # Label: 0 = normal (-), 1 = anomaly
        label = 0 if label_str == '-' else 1
        
        for tid, is_func in self.all_parsers:
            try:
                if is_func(content):
                    return self.event_vocab.get(f"Q{tid}", self.unknown_idx), label
            except: pass
        return self.event_vocab.get("Q_UNKNOWN", self.unknown_idx), label

parser = QwenLogRealTimeParser(event_vocab)

✓ Loaded 6663 templates


In [5]:
# Real-Time Detection Pipeline
class RealTimeAnomalyDetector:
    def __init__(self, model, parser, sequence_length, block_size, device):
        self.model = model
        self.parser = parser
        self.sequence_length = sequence_length
        self.block_size = block_size
        self.device = device
        self.event_buffer = []
        self.label_buffer = []
    
    def process_line(self, line):
        event_idx, label = self.parser.parse_line(line)
        if event_idx is None:
            return None, None
        
        self.event_buffer.append(event_idx)
        self.label_buffer.append(label)
        
        if len(self.event_buffer) >= self.sequence_length:
            seq = self.event_buffer[-self.sequence_length:]
            true_label = 1 if 1 in self.label_buffer[-self.sequence_length:] else 0
            
            with torch.no_grad():
                x = torch.tensor([seq], dtype=torch.long, device=self.device)
                pred = self.model(x).argmax(dim=1).item()
            
            if len(self.event_buffer) >= self.block_size:
                self.event_buffer = self.event_buffer[-self.sequence_length:]
                self.label_buffer = self.label_buffer[-self.sequence_length:]
            
            return pred, true_label
        return None, None

detector = RealTimeAnomalyDetector(model, parser, sequence_length, block_size, device)
print("✓ Real-time detector ready")

✓ Real-time detector ready


In [22]:
# FULL CHUNK1 BENCHMARK (70.4M lines)
# This will take several hours to complete

RAW_LOG_FILE = "./qwenlog_final/dataset/Thunderbird.log"
CHUNK1_LINES = 70_404_049  # Full Chunk1 size

print("="*70)
print("FULL THUNDERBIRD CHUNK1 BENCHMARK (70.4M lines)")
print("="*70)
print(f"Log file: {RAW_LOG_FILE}")
print(f"Total lines to process: {CHUNK1_LINES:,}")
print(f"Estimated time: ~17 hours (at 1,140 lines/sec)")
print("="*70)

FULL THUNDERBIRD CHUNK1 BENCHMARK (70.4M lines)
Log file: ./qwenlog_final/dataset/Thunderbird.log
Total lines to process: 70,404,049
Estimated time: ~17 hours (at 1,140 lines/sec)


In [ ]:
# Run Full Real-Time Benchmark
import gc
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

detector = RealTimeAnomalyDetector(model, parser, sequence_length, block_size, device)

y_true_all, y_pred_all = [], []
total_lines = 0
total_anomaly_lines = 0
unknown_count = 0

start_time = time.time()
last_report = start_time

print(f"Starting full benchmark at {time.strftime('%Y-%m-%d %H:%M:%S')}...")
print("Progress updates every 1M lines...")
print()

with open(RAW_LOG_FILE, 'r', encoding='utf-8', errors='ignore') as f:
    for line_num, line in enumerate(f):
        if line_num >= CHUNK1_LINES:
            break
        
        total_lines += 1
        
        # Track anomaly lines directly
        if not line.strip().startswith('-'):
            total_anomaly_lines += 1
        
        pred, true_label = detector.process_line(line)
        if pred is not None:
            y_pred_all.append(pred)
            y_true_all.append(true_label)
        
        # Progress report every 1M lines
        if (line_num + 1) % 1_000_000 == 0:
            elapsed = time.time() - start_time
            rate = total_lines / elapsed
            eta_hours = (CHUNK1_LINES - total_lines) / rate / 3600
            
            # Quick metrics
            curr_anomalies = sum(y_true_all)
            curr_pred_anomalies = sum(y_pred_all)
            
            print(f"[{time.strftime('%H:%M:%S')}] {total_lines/1e6:.1f}M / 70.4M lines "
                  f"({100*total_lines/CHUNK1_LINES:.1f}%) | "
                  f"Rate: {rate:,.0f} lines/s | ETA: {eta_hours:.1f}h | "
                  f"Anomaly seq: {curr_anomalies:,} true, {curr_pred_anomalies:,} pred")

total_time = time.time() - start_time
print()
print(f"✓ Benchmark completed at {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"✓ Total time: {total_time/3600:.2f} hours ({total_time:.0f} seconds)")
print(f"✓ Total lines: {total_lines:,}")
print(f"✓ Total sequences: {len(y_pred_all):,}")
print(f"✓ Throughput: {total_lines/total_time:,.0f} lines/sec")

Starting full benchmark at 2025-12-05 14:14:38...
Progress updates every 1M lines...

[14:29:07] 1.0M / 70.4M lines (1.4%) | Rate: 1,150 lines/s | ETA: 16.8h | Anomaly seq: 80 true, 60 pred
[14:43:39] 2.0M / 70.4M lines (2.8%) | Rate: 1,148 lines/s | ETA: 16.5h | Anomaly seq: 96,815 true, 96,755 pred
[14:58:05] 3.0M / 70.4M lines (4.3%) | Rate: 1,151 lines/s | ETA: 16.3h | Anomaly seq: 684,604 true, 684,298 pred
[15:12:28] 4.0M / 70.4M lines (5.7%) | Rate: 1,153 lines/s | ETA: 16.0h | Anomaly seq: 1,332,799 true, 1,332,413 pred
[15:26:47] 5.0M / 70.4M lines (7.1%) | Rate: 1,155 lines/s | ETA: 15.7h | Anomaly seq: 2,272,048 true, 2,271,662 pred
[15:41:13] 6.0M / 70.4M lines (8.5%) | Rate: 1,155 lines/s | ETA: 15.5h | Anomaly seq: 3,170,028 true, 3,169,632 pred
[15:56:31] 7.0M / 70.4M lines (9.9%) | Rate: 1,145 lines/s | ETA: 15.4h | Anomaly seq: 3,642,652 true, 3,641,964 pred
[16:11:11] 8.0M / 70.4M lines (11.4%) | Rate: 1,144 lines/s | ETA: 15.2h | Anomaly seq: 3,642,732 true, 3,641,96

In [ ]:
# Calculate Final Metrics
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

y_true = np.array(y_true_all)
y_pred = np.array(y_pred_all)

accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, zero_division=0)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)

print("="*70)
print("THUNDERBIRD CHUNK1 FULL BENCHMARK RESULTS")
print("="*70)
print(f"\n📊 Dataset Statistics:")
print(f"   Total lines processed: {total_lines:,}")
print(f"   Total sequences evaluated: {len(y_pred_all):,}")
print(f"   Raw anomaly lines: {total_anomaly_lines:,} ({100*total_anomaly_lines/total_lines:.2f}%)")

print(f"\n📈 Detection Results:")
print(f"   True anomaly sequences: {y_true.sum():,}")
print(f"   Predicted anomaly sequences: {y_pred.sum():,}")

print(f"\n🎯 Performance Metrics:")
print(f"   Accuracy:    {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   F1 Score:    {f1:.4f} ({f1*100:.2f}%)")
print(f"   Precision:   {precision:.4f} ({precision*100:.2f}%)")
print(f"   Recall:      {recall:.4f} ({recall*100:.2f}%)")

print(f"\n⏱️ Performance:")
print(f"   Total time: {total_time/3600:.2f} hours")
print(f"   Throughput: {total_lines/total_time:,.0f} lines/sec")

print(f"\n📋 Confusion Matrix:")
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
print(f"   TN (Normal→Normal):     {cm[0,0]:,}")
print(f"   FP (Normal→Anomaly):    {cm[0,1]:,}")
print(f"   FN (Anomaly→Normal):    {cm[1,0]:,}")
print(f"   TP (Anomaly→Anomaly):   {cm[1,1]:,}")
print("="*70)

In [ ]:
# Save Results to JSON
from datetime import datetime

results = {
    "dataset": "Thunderbird_Chunk1",
    "benchmark_type": "full_real_time",
    "timestamp": datetime.now().isoformat(),
    "total_lines": total_lines,
    "total_sequences": len(y_pred_all),
    "raw_anomaly_lines": total_anomaly_lines,
    "true_anomaly_sequences": int(y_true.sum()),
    "predicted_anomaly_sequences": int(y_pred.sum()),
    "metrics": {
        "accuracy": float(accuracy),
        "f1_score": float(f1),
        "precision": float(precision),
        "recall": float(recall)
    },
    "confusion_matrix": {
        "TN": int(cm[0,0]),
        "FP": int(cm[0,1]),
        "FN": int(cm[1,0]),
        "TP": int(cm[1,1])
    },
    "performance": {
        "total_time_seconds": total_time,
        "total_time_hours": total_time/3600,
        "throughput_lines_per_sec": total_lines/total_time
    },
    "model_info": {
        "vocab_size": len(event_vocab),
        "sequence_length": sequence_length,
        "block_size": block_size,
        "training_accuracy": metadata.get('final_accuracy', None)
    }
}

output_file = "./results/thunderbird_chunk1_full_benchmark.json"
with open(output_file, "w") as f:
    json.dump(results, f, indent=2)

print(f"✓ Results saved to {output_file}")